In [3]:
# evaluate.py
import os, re, json
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, precision_score, recall_score, multilabel_confusion_matrix,
    roc_curve, auc, precision_recall_curve
)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from lime.lime_text import LimeTextExplainer
from captum.attr import LayerIntegratedGradients

EMOTION_COLUMNS = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral"
]
NEUTRAL_IDX = EMOTION_COLUMNS.index("neutral")

MODEL_DIR = "./goemotions_model_v3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CSV_DIR = "./eval_outputs"
os.makedirs(CSV_DIR, exist_ok=True)

LIME_NUM_SAMPLES = 100

PLOTLY_FONT_COLOR = "#CBD5E1"
PLOTLY_TITLE_COLOR = "#E2E8F0"
PLOTLY_GRID_COLOR = "#262B45"
PLOTLY_ZERO_COLOR = "#64748B"


def style_fig(fig, title=None, height=480):
    if title:
        fig.update_layout(title=dict(text=title, font=dict(color=PLOTLY_TITLE_COLOR, size=14)))
    fig.update_layout(
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font=dict(color=PLOTLY_FONT_COLOR),
        height=height,
        margin=dict(l=10, r=10, t=50, b=10),
    )
    fig.update_xaxes(gridcolor=PLOTLY_GRID_COLOR, zerolinecolor=PLOTLY_ZERO_COLOR, color=PLOTLY_FONT_COLOR)
    fig.update_yaxes(gridcolor=PLOTLY_GRID_COLOR, zerolinecolor=PLOTLY_ZERO_COLOR, color=PLOTLY_FONT_COLOR)
    return fig


# ---- read train.py's saved config back, instead of re-hardcoding it ----
with open(os.path.join(MODEL_DIR, "metrics.json")) as f:
    TRAIN_META = json.load(f)
DATA_PATH = TRAIN_META["data_path"]
SAMPLE_SIZE = TRAIN_META["sample_size"]
MAX_LEN = TRAIN_META["max_len"]

if "full_dataset_size" in TRAIN_META:
    print(f"NOTE: model trained on {TRAIN_META['sample_size']}/"
          f"{TRAIN_META['full_dataset_size']} rows "
          f"({TRAIN_META.get('sample_fraction', 0):.1%} of the full dataset). "
          f"This is a compute tradeoff — treat it as a limitation when "
          f"interpreting the metrics below.")


def load_dataset(path):
    df = pd.read_csv(path)
    if "example_very_unclear" in df.columns:
        df = df[df["example_very_unclear"] == False]  # noqa: E712
    df = df[df[EMOTION_COLUMNS].sum(axis=1) > 0].reset_index(drop=True)
    for col in EMOTION_COLUMNS:
        df[col] = df[col].astype(int)
    return df


def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"\[NAME\]|\[RELIGION\]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def get_val_split():
    df = load_dataset(DATA_PATH)
    sample = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)
    texts = sample["text"].apply(clean_text).tolist()
    labels = sample[EMOTION_COLUMNS].values.tolist()
    _, val_texts, _, val_labels = train_test_split(texts, labels, test_size=0.15, random_state=42)
    return val_texts, np.array(val_labels)


print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()


def predict_proba(texts, batch_size=64):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, truncation=True, padding=True,
                         max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            logits = model(**enc).logits
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
    return np.vstack(all_probs)


print("Reproducing validation split & running predictions...")
val_texts, val_labels = get_val_split()
val_probs = predict_proba(val_texts)
val_preds = (val_probs >= 0.5).astype(int)

# 1. Per-label metrics
per_label_f1 = f1_score(val_labels, val_preds, average=None, zero_division=0)
per_label_prec = precision_score(val_labels, val_preds, average=None, zero_division=0)
per_label_rec = recall_score(val_labels, val_preds, average=None, zero_division=0)
support = val_labels.sum(axis=0)

metrics_df = pd.DataFrame({
    "label": EMOTION_COLUMNS, "f1": per_label_f1, "precision": per_label_prec,
    "recall": per_label_rec, "support": support
}).sort_values("f1", ascending=False)

print("\n=== Per-label metrics ===")
print(metrics_df.to_string(index=False))
metrics_df.to_csv(f"{CSV_DIR}/per_label_metrics.csv", index=False)

# ---- Interactive: per-label F1 ----
ordered = metrics_df.iloc[::-1]
fig_f1 = go.Figure(go.Bar(
    x=ordered["f1"], y=ordered["label"], orientation="h",
    marker=dict(color="#818CF8"),
    customdata=np.stack([ordered["precision"], ordered["recall"], ordered["support"]], axis=-1),
    hovertemplate=("<b>%{y}</b><br>F1: %{x:.3f}<br>Precision: %{customdata[0]:.3f}"
                   "<br>Recall: %{customdata[1]:.3f}<br>Support: %{customdata[2]:.0f}<extra></extra>"),
))
style_fig(fig_f1, "Per-label F1 score (sorted) — hover for precision/recall/support", height=700)
fig_f1.write_json(f"{CSV_DIR}/per_label_f1.json")

# 2. Confusion matrices
TOP_N_CM = 8
top_idx = np.argsort(-support)[:TOP_N_CM]
mcm = multilabel_confusion_matrix(val_labels, val_preds)

fig_cm = make_subplots(
    rows=2, cols=4, subplot_titles=[EMOTION_COLUMNS[i] for i in top_idx],
    horizontal_spacing=0.05, vertical_spacing=0.20,
)
for i, idx in enumerate(top_idx):
    r, c = i // 4 + 1, i % 4 + 1
    cm = mcm[idx]
    fig_cm.add_trace(go.Heatmap(
        z=cm, x=["Pred 0", "Pred 1"], y=["True 0", "True 1"],
        text=cm, texttemplate="%{text}", textfont=dict(color="white", size=13),
        colorscale="Blues", showscale=False, zmin=0,
        hovertemplate="%{y} / %{x}: %{z}<extra></extra>",
    ), row=r, col=c)
fig_cm.update_yaxes(autorange="reversed")
style_fig(fig_cm, f"Confusion matrices — top {TOP_N_CM} most frequent emotions (hover a cell for its count)", height=560)
fig_cm.update_annotations(font=dict(color=PLOTLY_TITLE_COLOR, size=12))
fig_cm.write_json(f"{CSV_DIR}/confusion_matrices.json")

# 3. ROC curves
fig_roc = go.Figure()
for idx in top_idx:
    fpr, tpr, _ = roc_curve(val_labels[:, idx], val_probs[:, idx])
    fig_roc.add_trace(go.Scatter(
        x=fpr, y=tpr, mode="lines", name=f"{EMOTION_COLUMNS[idx]} (AUC={auc(fpr, tpr):.2f})",
        hovertemplate="FPR: %{x:.2f}<br>TPR: %{y:.2f}<extra></extra>",
    ))
fig_roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
                              line=dict(dash="dash", color=PLOTLY_ZERO_COLOR),
                              showlegend=False, hoverinfo="skip"))
fig_roc.update_layout(xaxis_title="False positive rate", yaxis_title="True positive rate",
                       legend=dict(font=dict(size=10)))
style_fig(fig_roc, "ROC curves — most frequent emotions")
fig_roc.write_json(f"{CSV_DIR}/roc_curves.json")

# 4. Precision-Recall curves
fig_pr = go.Figure()
for idx in top_idx:
    prec, rec, _ = precision_recall_curve(val_labels[:, idx], val_probs[:, idx])
    fig_pr.add_trace(go.Scatter(
        x=rec, y=prec, mode="lines", name=EMOTION_COLUMNS[idx],
        hovertemplate="Recall: %{x:.2f}<br>Precision: %{y:.2f}<extra></extra>",
    ))
fig_pr.update_layout(xaxis_title="Recall", yaxis_title="Precision", legend=dict(font=dict(size=10)))
style_fig(fig_pr, "Precision-Recall curves — most frequent emotions")
fig_pr.write_json(f"{CSV_DIR}/pr_curves.json")

# 5. True vs predicted frequency 
true_freq = val_labels.sum(axis=0)
pred_freq = val_preds.sum(axis=0)
order2 = np.argsort(-true_freq)
labels_sorted = [EMOTION_COLUMNS[i] for i in order2]

fig_freq = go.Figure()
fig_freq.add_trace(go.Bar(x=labels_sorted, y=true_freq[order2], name="True", marker_color="#4C72B0",
                           hovertemplate="%{x}<br>True count: %{y}<extra></extra>"))
fig_freq.add_trace(go.Bar(x=labels_sorted, y=pred_freq[order2], name="Predicted", marker_color="#DD8452",
                           hovertemplate="%{x}<br>Predicted count: %{y}<extra></extra>"))
fig_freq.update_layout(barmode="group", xaxis_tickangle=-70, yaxis_title="Count in validation set")
style_fig(fig_freq, "True vs predicted label frequency", height=500)
fig_freq.write_json(f"{CSV_DIR}/true_vs_pred_freq.json")

print("\nInteractive charts (.json) + per-label metrics CSV saved to", CSV_DIR)

# 6. Faithfulness: LIME vs Integrated Gradients
_lime_explainer = LimeTextExplainer(class_names=EMOTION_COLUMNS)
_embedding_layer = model.distilbert.embeddings
_lig = LayerIntegratedGradients(
    lambda input_ids, attention_mask, target_idx: model(
        input_ids=input_ids, attention_mask=attention_mask).logits[:, target_idx],
    _embedding_layer,
)


def target_prob(text, target_idx):
    return float(predict_proba([text])[0, target_idx])


def mask_out(text, words):
    if not words:
        return text
    pattern = r'\b(' + '|'.join(re.escape(w) for w in words) + r')\b'
    return re.sub(r'\s+', ' ', re.sub(pattern, '', text, flags=re.IGNORECASE)).strip()


def lime_top_words(text, target_idx, k=3, num_samples=LIME_NUM_SAMPLES):
    exp = _lime_explainer.explain_instance(
        text, predict_proba, labels=(target_idx,), num_features=8, num_samples=num_samples)
    ranked = sorted(exp.as_list(label=target_idx), key=lambda x: -abs(x[1]))
    return [w for w, _ in ranked[:k]]


def ig_top_words(text, target_idx, k=3, steps=32):
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN, return_tensors="pt")
    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)
    baseline_ids = input_ids.clone()
    baseline_ids[:, 1:-1] = tokenizer.pad_token_id

    attributions = _lig.attribute(
        inputs=input_ids, baselines=baseline_ids,
        additional_forward_args=(attention_mask, target_idx), n_steps=steps,
    ).sum(dim=-1).squeeze(0)
    attributions = attributions / (attributions.norm() + 1e-8)

    tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0).tolist())
    words, scores, cur_word, cur_score = [], [], None, 0.0
    for tok, score in zip(tokens, attributions.tolist()):
        if tok in (tokenizer.cls_token, tokenizer.sep_token, tokenizer.pad_token):
            continue
        if tok.startswith("##"):
            cur_word += tok[2:]; cur_score += score
        else:
            if cur_word is not None:
                words.append(cur_word); scores.append(cur_score)
            cur_word, cur_score = tok, score
    if cur_word is not None:
        words.append(cur_word); scores.append(cur_score)

    ranked = sorted(zip(words, scores), key=lambda x: -abs(x[1]))
    return [w for w, _ in ranked[:k]]


def comprehensiveness(text, target_idx, top_words):
    return target_prob(text, target_idx) - target_prob(mask_out(text, top_words), target_idx)


def sufficiency(text, target_idx, top_words):
    kept_only = " ".join(top_words) if top_words else text
    return target_prob(text, target_idx) - target_prob(kept_only, target_idx)


N_FAITHFULNESS = 25
K_WORDS = 3
CONFIDENCE_THRESHOLD = 0.3
rng = np.random.RandomState(7)

candidate_order = rng.permutation(len(val_texts))

rows = []
print(f"\nRunning faithfulness evaluation "
      f"(target: {N_FAITHFULNESS} examples, preferring non-neutral labels)...")
for idx in candidate_order:
    if len(rows) >= N_FAITHFULNESS:
        break

    text = val_texts[idx]
    probs = predict_proba([text])[0]

    non_neutral_probs = probs.copy()
    non_neutral_probs[NEUTRAL_IDX] = -1.0
    best_non_neutral_idx = int(np.argmax(non_neutral_probs))

    if non_neutral_probs[best_non_neutral_idx] >= CONFIDENCE_THRESHOLD:
        target_idx = best_non_neutral_idx
    elif probs[NEUTRAL_IDX] >= CONFIDENCE_THRESHOLD:
        target_idx = NEUTRAL_IDX
    else:
        continue  # no confident prediction at all — skip, don't force it

    lime_words = lime_top_words(text, target_idx, k=K_WORDS)
    ig_words = ig_top_words(text, target_idx, k=K_WORDS)

    rows.append({
        "text": text[:60], "label": EMOTION_COLUMNS[target_idx],
        "lime_comprehensiveness": comprehensiveness(text, target_idx, lime_words),
        "lime_sufficiency": sufficiency(text, target_idx, lime_words),
        "ig_comprehensiveness": comprehensiveness(text, target_idx, ig_words),
        "ig_sufficiency": sufficiency(text, target_idx, ig_words),
    })
    if len(rows) % 5 == 0:
        print(f"  {len(rows)}/{N_FAITHFULNESS} done")

faith_df = pd.DataFrame(rows)
faith_df.to_csv(f"{CSV_DIR}/faithfulness_raw.csv", index=False)

neutral_count = int((faith_df["label"] == "neutral").sum())
print(f"\nCollected {len(faith_df)} examples "
      f"({neutral_count} neutral, {len(faith_df) - neutral_count} non-neutral) "
      f"across {faith_df['label'].nunique()} distinct labels.")

summary = faith_df[["lime_comprehensiveness", "lime_sufficiency",
                     "ig_comprehensiveness", "ig_sufficiency"]].mean()

print(f"\n=== Faithfulness summary (n={len(faith_df)}) ===")
print(f"  LIME — comprehensiveness: {summary['lime_comprehensiveness']:.3f}  "
      f"sufficiency drop: {summary['lime_sufficiency']:.3f}")
print(f"  IG   — comprehensiveness: {summary['ig_comprehensiveness']:.3f}  "
      f"sufficiency drop: {summary['ig_sufficiency']:.3f}")

# ---- Interactive: faithfulness comparison ----
labels2 = ["Comprehensiveness (higher=better)", "Sufficiency drop (lower=better)"]
lime_vals = [summary["lime_comprehensiveness"], summary["lime_sufficiency"]]
ig_vals = [summary["ig_comprehensiveness"], summary["ig_sufficiency"]]

fig_faith = go.Figure()
fig_faith.add_trace(go.Bar(x=labels2, y=lime_vals, name="LIME", marker_color="#DD8452",
                            hovertemplate="%{x}<br>LIME: %{y:.3f}<extra></extra>"))
fig_faith.add_trace(go.Bar(x=labels2, y=ig_vals, name="Integrated Gradients", marker_color="#55A868",
                            hovertemplate="%{x}<br>IG: %{y:.3f}<extra></extra>"))
fig_faith.update_layout(barmode="group")
style_fig(fig_faith, f"Faithfulness comparison (n={len(faith_df)}, {len(faith_df) - neutral_count} non-neutral)", height=460)
fig_faith.write_json(f"{CSV_DIR}/faithfulness_comparison.json")

print(f"\nDone. Faithfulness CSV + all interactive charts saved to {CSV_DIR}/")

NOTE: model trained on 15000/207814 rows (7.2% of the full dataset). This is a compute tradeoff — treat it as a limitation when interpreting the metrics below.
Loading model...
Reproducing validation split & running predictions...

=== Per-label metrics ===
         label       f1  precision   recall  support
       remorse 0.580000   0.432836 0.878788       33
       neutral 0.530794   0.429599 0.694352      602
     gratitude 0.530055   0.389558 0.829060      117
          love 0.487633   0.359375 0.758242       91
     amusement 0.452830   0.322581 0.759494       79
    admiration 0.412869   0.273050 0.846154      182
     curiosity 0.393736   0.254335 0.871287      101
        desire 0.384181   0.285714 0.586207       58
          fear 0.323810   0.232877 0.531250       32
     confusion 0.298077   0.191950 0.666667       93
         anger 0.278689   0.178010 0.641509      106
           joy 0.268000   0.163415 0.744444       90
      optimism 0.247573   0.157407 0.579545       88
